# Hello World Deep Learning in Medical Imaging

Code from the following [article](https://link.springer.com/article/10.1007/s10278-018-0079-6).

## Data prepration

The images come from the [GitHub repository](https://github.com/paras42/Hello_World_Deep_Learning/blob/9921a12c905c00a88898121d5dc538e3b524e520/Open_I_abd_vs_CXRs.zip) of the tutorial. The first time the notebook runs, they are downloaded and unzipped into `data/`, at the root of the repository (the folder is ignored by Git).

## Imports

In [ ]:
import math

import keras
from keras import applications
from keras import layers
from keras.models import Sequential
from keras.layers import Dropout, Flatten, Dense, GlobalAveragePooling2D
from keras.models import Model
from keras.optimizers import Adam

## Configuration

Paths, hyperparameters and the random seed are read from `config.toml`, at the root of the repository, instead of being written in the code. To run another experiment, copy that file, edit the copy, and pass it as `config_file` (with papermill: `-p config_file my-experiment.toml`).

In [ ]:
# Parameters: papermill can override them (papermill ... -p config_file my-experiment.toml)
config_file = '../config.toml'

In [ ]:
import tomllib
from pathlib import Path

with open(config_file, 'rb') as file:
    config = tomllib.load(file)

# Paths in config.toml are relative to the root of the repository, one level above this notebook
repository = Path('..')
dataset_dir = repository / config['dataset_directory']

# dimensions of our images
img_width = img_height = config['image_size']

# directory and image information
train_data_dir = dataset_dir / 'TRAIN'
validation_data_dir = dataset_dir / 'VAL'
test_data_dir = dataset_dir / 'TEST'

# epochs = number of passes of through training data
# batch_size = number images processed at same time
epochs = config['epochs']
batch_size = config['batch_size']
learning_rate = config['learning_rate']
augmentation_ranges = config['augmentation']

# One seed for Python, NumPy and TensorFlow: same initial weights, same shuffling
# and same augmentations from one run to the next
keras.utils.set_random_seed(config['seed'])

In [ ]:
import urllib.request
import zipfile

# Download and unzip the images, the first time only
if not dataset_dir.exists():
    archive, _ = urllib.request.urlretrieve(config['dataset_url'])
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(dataset_dir.parent)

## Build the model

Here, we use [Inception V3](https://keras.io/api/applications/inceptionv3/).

In [ ]:
# build the Inception V3 network, use pretrained weights from ImageNet
# remove top fully connected layers by include_top=False
base_model = applications.InceptionV3(weights='imagenet',
                                      include_top=False,
                                      input_shape=(img_width, img_height, 3))

### Add new layers on top of the model

In [ ]:
# build a classifier model to put on top of the convolutional model
# This consists of a global average pooling layer and a fully connected layer with 256 nodes
# Then apply dropout and sigmoid activation
model_top = Sequential()
model_top.add(keras.Input(shape=base_model.output_shape[1:]))
model_top.add(GlobalAveragePooling2D(data_format=None))
model_top.add(Dense(256, activation='relu'))
model_top.add(Dropout(0.5))
model_top.add(Dense(1, activation='sigmoid'))
model = Model(inputs=base_model.input, outputs=model_top(base_model.output))
# Compile model using Adam optimizer with common values and binary cross entropy loss
# Use low learning rate (lr) for transfer learning
# (Keras 3 renamed `lr` to `learning_rate` and dropped `decay`, which was 0 anyway.)
model.compile(optimizer=Adam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.999, epsilon=1e-08),
              loss='binary_crossentropy',
              metrics=['accuracy'])

Note that for classification tasks with greater than 2 classes (e.g., ImageNet has 1000 classes),
*categorical cross-entropy* is typically used as the loss function; for tasks with 2 classes (such as here), *binary cross-entropy* is used.

## Image Preprocessing and Augmentation

Preprocess and specify augmentation options, which include transformations and other variations to the image, which can help preempt overfitting or memorization of training data, and have shown to increase accuracy and generalization of CNNs.

While augmentation can be done inadvance, Keras has an image data generator, which can perform *on-the-fly* augmentation, such as rotations, translation, zoom, shearing, and flipping, just before they are fed to the network.

Keras 3 no longer has `ImageDataGenerator`. The same on-the-fly augmentation is done here by preprocessing layers, applied to the images as they are read.

In [ ]:
# ImageDataGenerator expressed shear in degrees and filled the borders with the
# nearest pixel; the Keras 3 layers below are set up to do the same.
augmentation = Sequential([
    # degrees of rotation, as a fraction of a full turn
    layers.RandomRotation(augmentation_ranges['rotation_degrees'] / 360, fill_mode='nearest'),
    # 0-1 range for zoom
    layers.RandomZoom(augmentation_ranges['zoom_range'], fill_mode='nearest'),
    # 0-1 range vertical and horizontal translation
    layers.RandomTranslation(augmentation_ranges['shift_range'], augmentation_ranges['shift_range'],
                             fill_mode='nearest'),
    # shear angle in degrees, as a horizontal shift per pixel of height
    layers.RandomShear(x_factor=math.tan(math.radians(augmentation_ranges['shear_degrees'])),
                       fill_mode='nearest'),
])
if augmentation_ranges['horizontal_flip']:
    augmentation.add(layers.RandomFlip('horizontal'))

# Rescale pixel values to 0-1 to aid CNN processing
rescale = layers.Rescaling(1./255)

## Defining the training and validation datasets

In [ ]:
# Directory, image size, batch size already specified above
# Label mode is set to 'binary' for a 2-class problem
# The dataset randomly shuffles and presents images in batches to the network
def read_images(directory):
    return keras.utils.image_dataset_from_directory(directory,
                                                    label_mode='binary',
                                                    image_size=(img_height, img_width),
                                                    batch_size=batch_size,
                                                    interpolation='nearest',
                                                    seed=config['seed'])

# Rescale and augment the training images as they are read
train_generator = read_images(train_data_dir).map(
    lambda images, labels: (augmentation(rescale(images), training=True), labels))

# Validation images are only rescaled: they must be the same at every epoch
validation_generator = read_images(validation_data_dir).map(
    lambda images, labels: (rescale(images), labels))

## Fitting the model

In [ ]:
# Fine-tune the pretrained Inception V3 model on the training dataset
# (a dataset knows its own length, so the number of steps per epoch is not needed)
history = model.fit(train_generator,
                    epochs=epochs,
                    validation_data=validation_generator)

### Plotting the model fitting convergence

In [ ]:
# import matplotlib library, and plot training cuve
import matplotlib.pyplot as plt

print(history.history.keys())
plt.figure()
plt.plot(history.history['accuracy'], 'orange', label='Training accuracy')
plt.plot(history.history['val_accuracy'], 'blue', label='Validation accuracy')
plt.plot(history.history['loss'], 'red', label='Training loss')
plt.plot(history.history['val_loss'], 'green', label='Validation loss')
plt.legend()
plt.show()

## Evaluating the model performance

In [ ]:
# import numpy and keras preprocessing libraries
import numpy as np
from keras.preprocessing import image

# load, resize, and display test images
img_path = test_data_dir / 'chest2.png'
img_path2 = test_data_dir / 'abd2.png'
img = image.load_img(img_path, target_size=(img_width, img_height))
img2 = image.load_img(img_path2, target_size=(img_width, img_height))
plt.imshow(img)
plt.show()

# convert image to numpy array, so Keras can render a prediction
img = image.img_to_array(img)

# expand array from 3 dimensions (height, width, channels) to 4 dimensions (batch size, height, width, channels)
# rescale pixel values to 0-1
x = np.expand_dims(img, axis=0) * 1./255

# get prediction on test image
score = model.predict(x)

print('Predicted:', score, 'Chest X-ray' if score < 0.5 else 'Abd X-ray')
# display and render a prediction for the 2nd image
plt.imshow(img2)
plt.show()
img2 = image.img_to_array(img2)
x = np.expand_dims(img2, axis=0) * 1./255
score2 = model.predict(x)
print('Predicted:', score2, 'Chest X-ray' if score2 < 0.5 else 'Abd X-ray')